# gen-gec-errant — Batch Run All Models (Colab)

Run the full **generate → GEC → ERRANT → analysis** pipeline for **all models**
in the registry across all 3 learner datasets.

**Use case:** Full paper reproduction. Runs sequentially through each model,
saving results to Google Drive after each. Designed for long Colab Pro sessions.

**Data source:** `phd-experimental-data/cefr-classification/data/splits/`  
**Models:** `_p/artificial-learners/models/` (fine-tuned) + HuggingFace Hub (native)  
**Output:** `_p/artificial-learners/generation-gec-errant/results/batch-<timestamp>/`  

**Runtime:** ~2–6 hours total on T4 GPU (depends on `MAX_SENTENCES`).

*Last updated: 2026-03-16*

## 0. Mount Google Drive & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%time
!pip install torch transformers errant spacy numpy scipy matplotlib pandas pyyaml accelerate -q
!python -m spacy download en_core_web_sm -q

In [ ]:
import sys
from pathlib import Path

GDRIVE = Path('/content/drive/MyDrive')
REPO_ON_DRIVE = GDRIVE / '_p/artificial-learners/generation-gec-errant'

# Option B: Clone from GitHub (uncomment if needed)
# !git clone https://github.com/YOUR_USER/gen-gec-errant.git /content/gen-gec-errant
# REPO_ON_DRIVE = Path('/content/gen-gec-errant')

if REPO_ON_DRIVE.exists():
    !cp -r "{REPO_ON_DRIVE}" /content/gen-gec-errant
    !pip install -e /content/gen-gec-errant -q
    print(f'Installed gen_gec_errant from {REPO_ON_DRIVE}')
else:
    print(f'WARNING: Repo not found at {REPO_ON_DRIVE}')
    sys.exit(1)

## 1. Configuration

Configure which models to run and dataset limits.

Set `MODELS_TO_RUN` to `None` to run all models, or pass a list of model keys to run a subset.

In [ ]:
from pathlib import Path
from datetime import datetime
import torch

# ── Paths ──────────────────────────────────────────────────────────────
GDRIVE = Path('/content/drive/MyDrive')

DATA_ROOT = GDRIVE / 'phd-experimental-data/cefr-classification/data/splits'
MODELS_ROOT = GDRIVE / '_p/artificial-learners/models'
OUTPUT_ROOT = GDRIVE / '_p/artificial-learners/generation-gec-errant/results'

TIMESTAMP = datetime.now().strftime('%Y%m%d-%H%M%S')
BATCH_RUN_DIR = OUTPUT_ROOT / f'batch-{TIMESTAMP}'

# ── Model Registry ─────────────────────────────────────────────────────
MODEL_REGISTRY = {
    # ── Native baselines ──
    'gpt2-small-native': {
        'params': '124M', 'hf_model_id': 'gpt2', 'model_family': 'gpt2',
        'gdrive_subpath': None, 'is_learner_tuned': False,
        'description': 'GPT-2 Small native (zero-shot baseline)',
    },
    # ── GPT-2 fine-tuned ──
    'ft-gpt2-small': {
        'params': '124M', 'hf_model_id': None, 'model_family': 'gpt2',
        'gdrive_subpath': '2026-02-23-model/gpt2-small-all-data-resume',
        'is_learner_tuned': True,
        'description': 'GPT-2 Small (124M) fine-tuned on EFCAMDAT all-data',
    },
    'ft-gpt2-medium': {
        'params': '355M', 'hf_model_id': None, 'model_family': 'gpt2',
        'gdrive_subpath': '2026-02-23-model/gpt2-medium-all-data-resume',
        'is_learner_tuned': True,
        'description': 'GPT-2 Medium (355M) fine-tuned on EFCAMDAT all-data',
    },
    'ft-gpt2-large': {
        'params': '774M', 'hf_model_id': None, 'model_family': 'gpt2',
        'gdrive_subpath': '2026-02-20-model/gpt2-large-all-data-20260222-092036',
        'is_learner_tuned': True,
        'description': 'GPT-2 Large (774M) fine-tuned on EFCAMDAT all-data',
    },
    # ── Pythia fine-tuned ──
    'ft-pythia-70m': {
        'params': '70M', 'hf_model_id': None, 'model_family': 'pythia',
        'gdrive_subpath': 'pythia/pythia-70m-all-data',
        'is_learner_tuned': True,
        'description': 'Pythia 70M fine-tuned on EFCAMDAT all-data',
    },
    'ft-pythia-160m': {
        'params': '160M', 'hf_model_id': None, 'model_family': 'pythia',
        'gdrive_subpath': 'pythia/pythia-160m-all-data',
        'is_learner_tuned': True,
        'description': 'Pythia 160M fine-tuned on EFCAMDAT all-data',
    },
    'ft-pythia-410m': {
        'params': '410M', 'hf_model_id': None, 'model_family': 'pythia',
        'gdrive_subpath': 'pythia/pythia-410m-all-data',
        'is_learner_tuned': True,
        'description': 'Pythia 410M fine-tuned on EFCAMDAT all-data',
    },
    'ft-pythia-1b': {
        'params': '1B', 'hf_model_id': None, 'model_family': 'pythia',
        'gdrive_subpath': 'pythia/pythia-1b-all-data',
        'is_learner_tuned': True,
        'description': 'Pythia 1B fine-tuned on EFCAMDAT all-data',
    },
    'ft-pythia-1.4b': {
        'params': '1.4B', 'hf_model_id': None, 'model_family': 'pythia',
        'gdrive_subpath': 'pythia/pythia-1.4b-all-data',
        'is_learner_tuned': True,
        'description': 'Pythia 1.4B fine-tuned on EFCAMDAT all-data',
    },
    # ── SmolLM2 fine-tuned ──
    'ft-smollm2-135m': {
        'params': '135M', 'hf_model_id': None, 'model_family': 'smollm2',
        'gdrive_subpath': 'smollm2/smollm2-135m-all-data',
        'is_learner_tuned': True,
        'description': 'SmolLM2 135M fine-tuned on EFCAMDAT all-data',
    },
    'ft-smollm2-360m': {
        'params': '360M', 'hf_model_id': None, 'model_family': 'smollm2',
        'gdrive_subpath': 'smollm2/smollm2-360m-all-data',
        'is_learner_tuned': True,
        'description': 'SmolLM2 360M fine-tuned on EFCAMDAT all-data',
    },
    'ft-smollm2-1.7b': {
        'params': '1.7B', 'hf_model_id': None, 'model_family': 'smollm2',
        'gdrive_subpath': 'smollm2/smollm2-1.7b-all-data',
        'is_learner_tuned': True,
        'description': 'SmolLM2 1.7B fine-tuned on EFCAMDAT all-data',
    },
}

# ══════════════════════════════════════════════════════════════════════
#  CONFIGURE WHICH MODELS TO RUN
# ══════════════════════════════════════════════════════════════════════
MODELS_TO_RUN = None  # None = all models; or e.g. ['ft-gpt2-small', 'ft-pythia-70m']
MAX_SENTENCES = None  # None = all data; set to 50 for quick test
# ══════════════════════════════════════════════════════════════════════

# ── Data files ─────────────────────────────────────────────────────────
# Same splits used across the project (cf. colab_train_bert.ipynb)
DATASETS = {
    'norm-CELVA-SP': {
        'path': DATA_ROOT / 'norm-CELVA-SP.csv',
        'description': 'Spanish L1 learner English (primary)',
    },
    'norm-EFCAMDAT-test': {
        'path': DATA_ROOT / 'norm-EFCAMDAT-test.csv',
        'description': 'In-domain learner English',
    },
    'norm-KUPA-KEYS': {
        'path': DATA_ROOT / 'norm-KUPA-KEYS.csv',
        'description': 'Cross-corpus learner English',
    },
}

# ── Pipeline params ────────────────────────────────────────────────────
GEC_MODEL = 'grammarly/coedit-large'
GEC_METHOD = 'dedicated'
BATCH_SIZE = 8
GEC_BATCH_SIZE = 16
SEED = 42

# Resolve model list
model_keys = MODELS_TO_RUN or list(MODEL_REGISTRY.keys())

# ── Device ──────────────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

print(f'\nBatch run dir: {BATCH_RUN_DIR}')
print(f'Models to run: {len(model_keys)}')
for k in model_keys:
    info = MODEL_REGISTRY[k]
    print(f'  {k:<25} {info["params"]:>5}  {info["description"]}')

print(f'\nDatasets: {len(DATASETS)}')
for name, info in DATASETS.items():
    print(f'  {name}: exists={info["path"].exists()}')

print(f'\nMax sentences: {MAX_SENTENCES or "all"}')
print(f'Total runs: {len(model_keys)} models x {len(DATASETS)} datasets = {len(model_keys) * len(DATASETS)}')

## 2. Run All Models

Iterates through each model, copies weights to local SSD, runs the pipeline for each dataset,
saves results to GDrive, and cleans up local model cache before moving to the next model.

In [ ]:
import logging
import time
import json
import shutil
import yaml
import gc

logging.basicConfig(level=logging.INFO, format='%(name)s | %(message)s')

from gen_gec_errant.pipeline import run_pipeline
from gen_gec_errant.pipeline.config import load_config_from_yaml


def resolve_model_path(model_key, model_info):
    """Get model path, copying from GDrive to local SSD if needed."""
    if model_info['gdrive_subpath']:
        gdrive_path = MODELS_ROOT / model_info['gdrive_subpath'] / 'final'
        local_cache = Path(f'/content/models/{model_key}')

        if not gdrive_path.exists():
            print(f'  WARNING: Model not found: {gdrive_path}')
            return None

        if not local_cache.exists():
            print(f'  Copying to local SSD...')
            shutil.copytree(gdrive_path, local_cache)

        return str(local_cache)
    else:
        return model_info['hf_model_id']


def cleanup_local_model(model_key):
    """Remove local model cache to free disk space."""
    local_cache = Path(f'/content/models/{model_key}')
    if local_cache.exists():
        shutil.rmtree(local_cache)
        print(f'  Cleaned up local cache: {local_cache}')


def write_config_yaml(model_key, model_info, model_path, dataset_name, data_path, output_dir):
    """Write a YAML pipeline config."""
    config_dir = BATCH_RUN_DIR / model_key / 'configs'
    config_dir.mkdir(parents=True, exist_ok=True)
    config_path = config_dir / f'{dataset_name}.yaml'

    config = {
        'data_loader': {
            'data_path': str(data_path),
            'text_column': 'text',
            'max_sentences': MAX_SENTENCES,
            'min_words': 10, 'max_words': 500,
            'prompt_ratio': 0.5, 'min_prompt_words': 5,
        },
        'generation': {
            'max_new_tokens': 50, 'min_new_tokens': 10,
            'temperature': 1.0, 'top_k': 50, 'top_p': 0.95,
            'do_sample': True, 'repetition_penalty': 1.2,
        },
        'gec': {
            'method': GEC_METHOD, 'model_id': GEC_MODEL,
            'batch_size': GEC_BATCH_SIZE, 'device': 'auto',
        },
        'annotation': {'lang': 'en'},
        'analysis': {'skip_plots': False, 'top_n_error_types': 10},
        'models': [{
            'name': model_key,
            'hf_model_id': model_path,
            'model_family': model_info['model_family'],
            'is_learner_tuned': model_info.get('is_learner_tuned', False),
        }],
        'batch_size': BATCH_SIZE,
        'device': 'auto',
        'seed': SEED,
        'output_dir': str(output_dir),
        'skip_plots': False,
    }

    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
    return config_path

In [ ]:
%%time

BATCH_RUN_DIR.mkdir(parents=True, exist_ok=True)

# Track progress across all models
progress_log = []
t_batch_start = time.time()

for model_idx, model_key in enumerate(model_keys, 1):
    model_info = MODEL_REGISTRY[model_key]

    print('\n' + '#' * 70)
    print(f'# MODEL {model_idx}/{len(model_keys)}: {model_key} ({model_info["params"]})')
    print(f'# {model_info["description"]}')
    print('#' * 70)

    # Resolve model path (copy to local SSD if fine-tuned)
    model_path = resolve_model_path(model_key, model_info)
    if model_path is None:
        progress_log.append({'model': model_key, 'status': 'skipped', 'reason': 'model not found'})
        continue

    model_results = {}

    for ds_name, ds_info in DATASETS.items():
        print(f'\n  --- {ds_name} ---')

        if not ds_info['path'].exists():
            print(f'  SKIPPED: file not found')
            continue

        output_dir = BATCH_RUN_DIR / model_key / ds_name
        raw_results = output_dir / 'raw_results.json'

        if raw_results.exists():
            print(f'  Already completed. Skipping.')
            continue

        config_path = write_config_yaml(
            model_key, model_info, model_path,
            ds_name, ds_info['path'], output_dir,
        )

        t0 = time.time()
        config = load_config_from_yaml(str(config_path))
        summaries, comparison = run_pipeline(config)
        elapsed = time.time() - t0

        model_results[ds_name] = {'summaries': summaries, 'elapsed': elapsed}
        print(f'  Completed in {elapsed / 60:.1f} min')

        # Free GPU memory between datasets
        if device == 'cuda':
            torch.cuda.empty_cache()

    # Build cross-dataset summary for this model
    cross_summary = {}
    for ds_name in DATASETS:
        output_dir = BATCH_RUN_DIR / model_key / ds_name
        entry = {'dataset': ds_name}

        model_summary_path = output_dir / f'{model_key}_summary.json'
        baseline_path = output_dir / 'learner_baseline_summary.json'

        if model_summary_path.exists():
            with open(model_summary_path) as f:
                entry['model'] = json.load(f)
        if baseline_path.exists():
            with open(baseline_path) as f:
                entry['learner_baseline'] = json.load(f)

        cross_summary[ds_name] = entry

    summary_path = BATCH_RUN_DIR / model_key / 'cross_dataset_summary.json'
    with open(summary_path, 'w') as f:
        json.dump(cross_summary, f, indent=2, default=str)

    progress_log.append({'model': model_key, 'status': 'completed', 'datasets': len(model_results)})

    # Cleanup local model cache to free disk
    cleanup_local_model(model_key)
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()

total_elapsed = time.time() - t_batch_start
print(f'\n{"=" * 70}')
print(f'BATCH RUN COMPLETE in {total_elapsed / 60:.1f} min ({total_elapsed / 3600:.1f} hours)')
print(f'Results: {BATCH_RUN_DIR}')
print(f'{"=" * 70}')

## 3. Aggregate Cross-Model Summary

In [ ]:
import pandas as pd

ds_display = {
    'norm-CELVA-SP': 'CELVA-SP',
    'norm-EFCAMDAT-test': 'EFCAMDAT-test',
    'norm-KUPA-KEYS': 'KUPA-KEYS',
}

# Build a table: rows=models, cols=dataset metrics
rows = []

for model_key in model_keys:
    summary_path = BATCH_RUN_DIR / model_key / 'cross_dataset_summary.json'
    if not summary_path.exists():
        continue

    with open(summary_path) as f:
        cross = json.load(f)

    info = MODEL_REGISTRY[model_key]
    row = {
        'model': model_key,
        'params': info['params'],
        'family': info['model_family'],
        'learner_tuned': info.get('is_learner_tuned', False),
    }

    for ds_name, ds_data in cross.items():
        ds_short = ds_display.get(ds_name, ds_name)
        m = ds_data.get('model', {})
        row[f'{ds_short}_ppl'] = m.get('ppl_mean')
        row[f'{ds_short}_err_rate'] = m.get('error_rate')
        row[f'{ds_short}_err_sent'] = m.get('avg_errors_per_sentence')

    rows.append(row)

if rows:
    df = pd.DataFrame(rows)
    print('Cross-Model Summary (Perplexity):')
    ppl_cols = ['model', 'params', 'family', 'learner_tuned'] + [c for c in df.columns if '_ppl' in c]
    print(df[ppl_cols].to_string(index=False))

    print('\nCross-Model Summary (Errors per Sentence):')
    err_cols = ['model', 'params'] + [c for c in df.columns if '_err_sent' in c]
    print(df[err_cols].to_string(index=False))

    # Save
    df.to_csv(BATCH_RUN_DIR / 'cross_model_summary.csv', index=False)
    print(f'\nSaved: {BATCH_RUN_DIR / "cross_model_summary.csv"}')
else:
    print('No completed results found.')

## 4. Cross-Model Visualizations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if not rows:
    print('No results to plot.')
else:
    # ── Perplexity by Model ────────────────────────────────────────────
    fig, axes = plt.subplots(1, len(ds_display), figsize=(6 * len(ds_display), 6))

    for ax, (ds_name, ds_short) in zip(axes, ds_display.items()):
        col = f'{ds_short}_ppl'
        valid = df[df[col].notna()].sort_values(col)

        colors = ['#DD8452' if lt else '#4C72B0' for lt in valid['learner_tuned']]
        ax.barh(valid['model'], valid[col], color=colors)
        ax.set_xlabel('Mean Perplexity')
        ax.set_title(ds_short)
        ax.invert_yaxis()

    fig.suptitle('Perplexity by Model and Dataset', fontsize=14)
    plt.tight_layout()
    fig.savefig(str(BATCH_RUN_DIR / 'perplexity_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # ── Error Rate by Model ───────────────────────────────────────────
    fig, axes = plt.subplots(1, len(ds_display), figsize=(6 * len(ds_display), 6))

    for ax, (ds_name, ds_short) in zip(axes, ds_display.items()):
        col = f'{ds_short}_err_sent'
        valid = df[df[col].notna()].sort_values(col)

        colors = ['#DD8452' if lt else '#4C72B0' for lt in valid['learner_tuned']]
        ax.barh(valid['model'], valid[col], color=colors)
        ax.set_xlabel('Avg Errors per Sentence')
        ax.set_title(ds_short)
        ax.invert_yaxis()

    fig.suptitle('Error Rate by Model and Dataset\n(orange=learner-tuned, blue=native)', fontsize=14)
    plt.tight_layout()
    fig.savefig(str(BATCH_RUN_DIR / 'error_rate_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Plots saved to {BATCH_RUN_DIR}')

## 5. Progress & Diagnostics

In [ ]:
# Show progress log
print('Progress Log:')
print(f'{"Model":<25} {"Status":<12} {"Details"}')
print(f'{"-"*25} {"-"*12} {"-"*20}')
for entry in progress_log:
    details = entry.get('reason', f'{entry.get("datasets", 0)} datasets')
    print(f'{entry["model"]:<25} {entry["status"]:<12} {details}')

# Save progress log
log_path = BATCH_RUN_DIR / 'progress_log.json'
with open(log_path, 'w') as f:
    json.dump(progress_log, f, indent=2)
print(f'\nProgress log saved: {log_path}')

# Save batch config
batch_config = {
    'models_run': model_keys,
    'max_sentences': MAX_SENTENCES,
    'batch_size': BATCH_SIZE,
    'gec_batch_size': GEC_BATCH_SIZE,
    'gec_model': GEC_MODEL,
    'seed': SEED,
    'timestamp': TIMESTAMP,
    'device': device,
    'gpu': torch.cuda.get_device_name() if device == 'cuda' else None,
}
with open(BATCH_RUN_DIR / 'batch_config.yaml', 'w') as f:
    yaml.dump(batch_config, f, default_flow_style=False)

print(f'\n=== All artifacts saved to {BATCH_RUN_DIR} ===')